# 01 — Data Audit and Cleaning

This notebook performs the full audit of the four raw datasets that ship with this
repository (`results.csv`, `goalscorers.csv`, `shootouts.csv`, `former_names.csv`),
then runs the reproducible cleaning pipeline in `src/data_cleaning.py` and writes
validated, analysis-ready files to `data/processed/`.

**Goals of this notebook**
1. Confirm the exact shape, dtypes, date ranges, and missing-value profile of each raw file.
2. Confirm `results.csv` contains at least 49,000 match records (a portfolio claim this
   project needs to back up with evidence, not assertion).
3. Identify and document duplicate rows (exact and conflicting).
4. Run the cleaning pipeline and verify the documented data-quality contract.
5. Surface known risks: historical team names, neutral-venue flags, and the
   presence of *scheduled/unplayed* 2026 World Cup fixtures with no recorded score.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RAW = PROJECT_ROOT / "data" / "raw"


## 1. Raw file inventory

In [2]:
raw_files = ["results.csv", "goalscorers.csv", "shootouts.csv", "former_names.csv"]
frames = {f.split(".")[0]: pd.read_csv(RAW / f) for f in raw_files}
for name, df in frames.items():
    print(f"{name}: shape={df.shape}")


results: shape=(49287, 9)
goalscorers: shape=(47601, 8)
shootouts: shape=(675, 5)
former_names: shape=(36, 4)


### results.csv — columns, dtypes, and a confirmed row count

In [3]:
results = frames["results"]
print(results.dtypes)
print()
print(f"Total rows in results.csv: {len(results):,}")
assert len(results) >= 49000, "Expected at least 49,000 match rows"
print("Confirmed: results.csv contains 49,000+ rows.")
results.head()


date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
neutral          bool
dtype: object

Total rows in results.csv: 49,287
Confirmed: results.csv contains 49,000+ rows.


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


### Date range, missing values, duplicates

In [4]:
results_dates = pd.to_datetime(results["date"])
print("Date range:", results_dates.min().date(), "to", results_dates.max().date())
print()
print("Missing values per column:")
print(results.isna().sum())
print()
print("Exact duplicate rows:", results.duplicated().sum())
print()
dup_key = results.duplicated(subset=["date", "home_team", "away_team"], keep=False)
print("Rows sharing (date, home_team, away_team) -- NOT necessarily exact duplicates:", dup_key.sum())
results[dup_key].sort_values(["date", "home_team"])


Date range: 1872-11-30 to 2026-06-27

Missing values per column:
date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64

Exact duplicate rows: 0

Rows sharing (date, home_team, away_team) -- NOT necessarily exact duplicates: 2


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
9637,1974-02-17,Tahiti,New Caledonia,2.0,1.0,Friendly,Papeete,Tahiti,False
9638,1974-02-17,Tahiti,New Caledonia,1.0,2.0,Friendly,Papeete,Tahiti,False


These two Tahiti vs New Caledonia rows on 1974-02-17 share the same teams and date but
are genuinely two different matches played at the same venue on the same day (a feature
of small island-tournament schedules at the time, not a data error). They are **not**
dropped as duplicates -- only exact full-row duplicates are removed automatically;
conflicting partial duplicates are surfaced for manual review (see
`outputs/tables/conflicting_duplicate_matches.csv`).

### Unique teams and tournaments

In [5]:
all_teams = set(results["home_team"]).union(set(results["away_team"]))
print("Unique teams (home or away):", len(all_teams))
print("Unique tournament names:", results["tournament"].nunique())
print()
wc_like = sorted([t for t in results["tournament"].unique() if "world cup" in t.lower()])
print("Tournament names containing the phrase 'World Cup':")
for t in wc_like:
    print(" -", t, f"({(results['tournament']==t).sum()} matches)")


Unique teams (home or away): 333
Unique tournament names: 193

Tournament names containing the phrase 'World Cup':
 - FIFA World Cup (1036 matches)
 - FIFA World Cup qualification (8771 matches)
 - Viva World Cup (60 matches)


**Risk identified:** only `FIFA World Cup` is the men's final tournament. `FIFA World Cup
qualification` and `Viva World Cup` (an unofficial competition for unrecognized/minority
nations) both contain the phrase "World Cup" but must be excluded from the World Cup
analysis -- this is handled explicitly in `src/feature_engineering.py`.

### Risk: unplayed / scheduled fixtures (the missing scores)

In [6]:
null_score_rows = results[results["home_score"].isna()]
print(f"Rows with no recorded score: {len(null_score_rows)}")
print("Tournament breakdown of these rows:")
print(null_score_rows["tournament"].value_counts())
print()
print("Date range of these rows:", pd.to_datetime(null_score_rows["date"]).min(), "to", pd.to_datetime(null_score_rows["date"]).max())


Rows with no recorded score: 72
Tournament breakdown of these rows:
tournament
FIFA World Cup    72
Name: count, dtype: int64

Date range of these rows: 2026-06-11 00:00:00 to 2026-06-27 00:00:00


All 72 rows with missing scores are **2026 FIFA World Cup fixtures that have not been
played yet** (the dataset was exported while the 2026 tournament was still upcoming).
Per the project brief, we do **not** fabricate or impute 2026 results -- these rows are
dropped from match-level analysis entirely. The 2026 tournament is excluded from every
downstream World Cup table and chart in this project.

### Risk: neutral venues, historical names, penalty shootouts

In [7]:
print("neutral column dtype:", results["neutral"].dtype, "| values:", results["neutral"].unique())
print()
former_names = frames["former_names"]
print(f"former_names.csv documents {len(former_names)} historical name changes, e.g.:")
former_names.head(8)


neutral column dtype: bool | values: [False  True]

former_names.csv documents 36 historical name changes, e.g.:


,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04
2,Curaçao,Netherlands Antilles,1957-03-03,2010-10-10
3,Czechoslovakia,Bohemia,1903-04-05,1919-01-01
4,Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01
5,Czechoslovakia,Representation of Czechs and Slovaks,1993-03-24,1993-11-17
6,DR Congo,Belgian Congo,1948-05-25,1956-01-02
7,DR Congo,Congo-Léopoldville,1963-04-12,1964-07-19


**Decision:** historical national-team identities (e.g. Czechoslovakia, Zaire, Burma)
are kept **separate** from their modern successors. Forcibly merging them would combine
different squads, eras, and in some cases different sovereign states, which would distort
win-rate and dominance metrics. `former_names.csv` is preserved as a documented reference
table only -- see `src/data_cleaning.py` module docstring for the full reasoning.

## 2. goalscorers.csv and shootouts.csv audits

In [8]:
gs = frames["goalscorers"]
so = frames["shootouts"]

print("goalscorers.csv:", gs.shape, "| date range:", gs['date'].min(), "to", gs['date'].max())
print(gs.isna().sum())
print()
print("shootouts.csv:", so.shape, "| date range:", so['date'].min(), "to", so['date'].max())
print(so.isna().sum())


goalscorers.csv: (47601, 8) | date range: 1916-07-02 to 2026-03-31
date           0
home_team      0
away_team      0
team           0
scorer        48
minute       256
own_goal       0
penalty        0
dtype: int64

shootouts.csv: (675, 5) | date range: 1967-08-22 to 2026-03-31
date               0
home_team          0
away_team          0
winner             0
first_shooter    429
dtype: int64


In [9]:
# Join feasibility check: can goalscorers/shootouts be reliably joined to results
# on (date, home_team, away_team)?
results_dt = results.copy()
results_dt["date"] = pd.to_datetime(results_dt["date"])
gs_dt = gs.copy(); gs_dt["date"] = pd.to_datetime(gs_dt["date"])
so_dt = so.copy(); so_dt["date"] = pd.to_datetime(so_dt["date"])

key_res = set(zip(results_dt["date"], results_dt["home_team"], results_dt["away_team"]))
key_gs = set(zip(gs_dt["date"], gs_dt["home_team"], gs_dt["away_team"]))
key_so = set(zip(so_dt["date"], so_dt["home_team"], so_dt["away_team"]))

print(f"goalscorers match-keys found in results.csv: {len(key_gs & key_res)} / {len(key_gs)}")
print(f"shootouts match-keys found in results.csv:    {len(key_so & key_res)} / {len(key_so)}")


goalscorers match-keys found in results.csv: 15440 / 15440
shootouts match-keys found in results.csv:    674 / 675


Both supplementary tables join cleanly onto `results.csv` (100% and 99.85% match-key
coverage respectively), so they are safe to use for the goalscorer and shootout
supplementary analyses in notebook 03.

## 3. Run the cleaning pipeline

In [10]:
from data_cleaning import run_cleaning_pipeline, find_conflicting_duplicates

cleaned = run_cleaning_pipeline()
for name, df in cleaned.items():
    print(f"{name}: {df.shape}")

print()
print("Cleaning log:", cleaned["results"].attrs.get("cleaning_log"))


results: (49215, 9)
goalscorers: (47601, 10)
shootouts: (675, 5)
former_names: (36, 4)

Cleaning log: {'rows_in_raw': 49287, 'rows_with_unparseable_dates_dropped': 0, 'exact_duplicate_rows_dropped': 0, 'unplayed_or_scheduled_rows_dropped': 72, 'rows_remaining_played_matches': 49215}


In [11]:
conflicts = find_conflicting_duplicates(cleaned["results"])
print(f"Conflicting (same date/home/away, different other fields) rows: {len(conflicts)}")
conflicts


Conflicting (same date/home/away, different other fields) rows: 2


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
9635,1974-02-17,Tahiti,New Caledonia,1,2,Friendly,Papeete,Tahiti,False
9636,1974-02-17,Tahiti,New Caledonia,2,1,Friendly,Papeete,Tahiti,False


## 4. Audit summary

| Check | Result |
|---|---|
| `results.csv` rows | 49,287 raw -> 49,215 played matches after dropping 72 unplayed 2026 fixtures |
| Exact duplicate rows | 0 |
| Conflicting partial duplicates | 2 (legitimate same-day, same-opponent island fixtures -- kept) |
| Missing values | Only `home_score`/`away_score`, all on unplayed 2026 fixtures |
| Unique teams | 333 (including non-FIFA regional/sub-national entities -- see notebook 04 for how this is handled) |
| Unique tournament names | 193 |
| World Cup final-tournament rows | 1,036 across 1930-2026 (2026 excluded from analysis: 0 played matches) |
| Date range | 1872-11-30 to 2026-06-27 (scheduled) / 2026-03-31 (last played match) |
| Negative scores | 0 |
| Cleaned files written | `data/processed/results_clean.csv`, `goalscorers_clean.csv`, `shootouts_clean.csv`, `former_names_clean.csv` |

Cleaning is complete and validated. Notebook 02 proceeds to exploratory analysis of all
international matches; notebook 03 isolates and analyzes FIFA World Cup matches from 2002
onward.